# Auditoría del Random Forest oficial — ENEMDU 2025

**Versión corregida con ZIP original.** Este notebook verifica el RF histórico usando `pred_rf.npy`, `split_test_idx.npy` y `adultos_mayores_con_cluster.csv` del ZIP privado. Calcula sensibilidad (recall) de todas las clases y falsos negativos. **No vuelve a entrenar** ni reemplaza la evaluación histórica con el resultado experimental 0,5504.

No es posible recuperar la sensibilidad a diferentes umbrales, el Brier o el log-loss desde etiquetas discretas: faltan las probabilidades originales. Mantenga el ZIP privado fuera de GitHub.

In [ ]:
%pip -q install numpy pandas scikit-learn openpyxl
import io, json, zipfile, hashlib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, precision_recall_fscore_support


## 1. Ubicar el ZIP privado
En Google Colab súbelo cuando aparezca el cuadro de selección. En Jupyter local, escribe la ruta exacta. El código no extrae la base al disco.

In [ ]:
ZIP_PRIVADO = Path('/content/enedmu_adultos_mayores.zip')
if not ZIP_PRIVADO.exists():
    try:
        from google.colab import files
        subidos = files.upload()
        if 'enedmu_adultos_mayores.zip' not in subidos: raise ValueError('Selecciona exactamente enedmu_adultos_mayores.zip')
        ZIP_PRIVADO = Path('enedmu_adultos_mayores.zip')
    except ImportError:
        raise FileNotFoundError('Ajusta ZIP_PRIVADO a la ruta del ZIP original.')
print('ZIP detectado:', ZIP_PRIVADO.name)


In [ ]:
with zipfile.ZipFile(ZIP_PRIVADO) as z:
    def leer(archivo):
        encontrados = [f.filename for f in z.infolist() if not f.is_dir() and f.filename.endswith('/' + archivo)]
        if len(encontrados) != 1: raise ValueError('Ausente/duplicado: ' + archivo)
        return z.read(encontrados[0])
    data = pd.read_csv(io.BytesIO(leer('adultos_mayores_con_cluster.csv')))
    ix = np.load(io.BytesIO(leer('split_test_idx.npy')), allow_pickle=False)
    bytes_rf = leer('pred_rf.npy')
    ypred = np.load(io.BytesIO(bytes_rf), allow_pickle=False)
if len(data)!=45959 or len(ix)!=13788 or len(ypred)!=13788 or len(np.unique(ix))!=13788: raise ValueError('Forma de insumos incompatible')
ytrue = data.iloc[ix]['cluster'].astype(int).to_numpy()
metricas = dict(accuracy=accuracy_score(ytrue, ypred), f1_macro=f1_score(ytrue, ypred, average='macro'), f1_ponderado=f1_score(ytrue, ypred, average='weighted'), kappa=cohen_kappa_score(ytrue, ypred))
assert abs(metricas['accuracy']-0.5308)<0.00005 and abs(metricas['f1_macro']-0.4588)<0.00005
print('RF oficial verificado:', json.dumps(metricas, indent=2, ensure_ascii=False))
print('SHA-256 de pred_rf.npy:', hashlib.sha256(bytes_rf).hexdigest())


## 2. Sensibilidad del clasificador oficial (Tabla A2)
Sensibilidad = VP/(VP+FN). Esta tabla utiliza las **predicciones originales del TEST** para describir el comportamiento por clase; no equivale a un análisis de cambio de umbral.

In [ ]:
C = confusion_matrix(ytrue, ypred, labels=[1,2,3,4])
precision, recall, f1, n = precision_recall_fscore_support(ytrue, ypred, labels=[1,2,3,4], zero_division=0)
tabla_A2 = pd.DataFrame([dict(Segmento=i+1, Casos=int(n[i]), Verdaderos_positivos=int(C[i,i]), Falsos_negativos=int(n[i]-C[i,i]), Sensibilidad_porcentaje=round(100*recall[i],2), Precision_porcentaje=round(100*precision[i],2), F1=round(float(f1[i]),4)) for i in range(4)])
display(tabla_A2)
print('Matriz de confusión (filas reales, columnas predichas):\n',C)
print('Falsos negativos del segmento 2 dirigidos a los segmentos 1, 3 y 4:', int(C[1,0]), int(C[1,2]), int(C[1,3]))


In [ ]:
salidas=Path('/content/resultados_rf_oficial') if Path('/content').exists() else Path.cwd()/'resultados_rf_oficial'
salidas.mkdir(parents=True,exist_ok=True)
with pd.ExcelWriter(salidas/'Tabla_A2_RF_sensibilidad_OFICIAL.xlsx') as w:
    tabla_A2.to_excel(w,sheet_name='RF_TEST_original',index=False)
    pd.DataFrame(C,index=['Real_1','Real_2','Real_3','Real_4'],columns=['Pred_1','Pred_2','Pred_3','Pred_4']).to_excel(w,sheet_name='Matriz_confusion')
(salidas/'auditoria_rf.json').write_text(json.dumps({'metricas':metricas,'sha256_pred_rf':hashlib.sha256(bytes_rf).hexdigest(),'observacion':'Las métricas de log-loss, Brier y los cambios de umbral no son recalculables desde pred_rf.npy'},indent=2,ensure_ascii=False),encoding='utf8')
print('Exportado a',salidas)
try:
    from google.colab import files
    files.download(str(salidas/'Tabla_A2_RF_sensibilidad_OFICIAL.xlsx'))
except ImportError: pass
